# Drought Prediction Model
## File: models/train_drought.ipynb

**Run karo:** Kernel → Restart & Run All
**Output:** `drought_model.pkl` → Copy karo `backend/` folder mein

## Step 1 — Libraries

In [ ]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, f1_score, roc_auc_score)
print("Ready!")

Ready!


## Step 2 — Dataset + Train + Save

In [2]:
def generate_drought_data(n=5000, seed=44):
    
    #Drought dataset for Indian agricultural regions.
    #Key factors: rainfall deficit, high temperature, low soil moisture.
    
    #Real sources:
    #- IMD drought monitoring: https://mausam.imd.gov.in
    #- SPI data: https://imdpune.gov.in
    
    np.random.seed(seed)
    
    rain_deficit   = np.random.uniform(-200, 100, n)
    temp_anomaly   = np.random.uniform(-2, 5, n)
    soil_moisture  = np.random.uniform(0.05, 0.8, n)
    evapotrans     = np.random.uniform(2, 12, n)
    groundwater    = np.random.uniform(1, 30, n)
    ndvi           = np.random.uniform(0.05, 0.8, n)
    spi_3          = np.random.uniform(-3, 3, n)
    dry_days       = np.random.uniform(0, 90, n)
    crop_stress    = np.random.uniform(0, 1, n)
    month          = np.random.randint(1, 13, n)
    humidity       = np.random.uniform(20, 90, n)
    
    p  = 0.05
    p += 0.25 * (-rain_deficit / 200).clip(0, 1)
    p += 0.20 * (temp_anomaly / 5).clip(0, 1)
    p += 0.15 * (1 - soil_moisture)
    p += 0.12 * (dry_days / 90)
    p += 0.10 * (-spi_3 / 3).clip(0, 1)
    p += 0.08 * crop_stress
    p += 0.05 * (groundwater / 30)
    p += np.where(np.isin(month, [3,4,5,11,12,1]), 0.05, 0)
    p += np.random.normal(0, 0.04, n)
    p  = p.clip(0, 1)
    drought = (p > np.percentile(p, 80)).astype(int)
    
    df = pd.DataFrame({
        'rainfall_deficit':rain_deficit.round(2),
        'temperature_anomaly':temp_anomaly.round(2),
        'soil_moisture':soil_moisture.round(3),
        'evapotranspiration':evapotrans.round(2),
        'groundwater_level':groundwater.round(2),
        'ndvi':ndvi.round(3),
        'spi_3month':spi_3.round(3),
        'consecutive_dry_days':dry_days.round(0),
        'crop_water_stress':crop_stress.round(3),
        'month':month,'humidity':humidity.round(1),
        'drought':drought
    })
    return df

FEATURES = [
    'rainfall_deficit','temperature_anomaly','soil_moisture','evapotranspiration',
    'groundwater_level','ndvi','spi_3month','consecutive_dry_days',
    'crop_water_stress','month','humidity',
    'water_stress_index','vegetation_stress','aridity_index'
]

def engineer(df):
    df = df.copy()
    df['water_stress_index'] = df['evapotranspiration'] / (df['soil_moisture'] + 0.01)
    df['vegetation_stress']  = (1 - df['ndvi']) * df['temperature_anomaly'].clip(0)
    df['aridity_index']      = df['consecutive_dry_days'] * df['temperature_anomaly'].clip(0)
    return df

df     = generate_drought_data()
df_eng = engineer(df)
X      = df_eng[FEATURES]; y = df_eng['drought']
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

model = GradientBoostingClassifier(n_estimators=150,max_depth=5,
    learning_rate=0.1,random_state=42)
model.fit(X_tr, y_tr)
pred  = model.predict(X_te)
proba = model.predict_proba(X_te)[:,1]

print("DROUGHT MODEL RESULTS")
print(f"F1-Score : {f1_score(y_te, pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_te, proba):.4f}")
print()
print(classification_report(y_te, pred, target_names=['No Drought','Drought']))

with open('drought_model.pkl','wb') as f:
    pickle.dump({'model':model,'features':FEATURES,'version':'1.0','disaster':'drought'},f)
print("drought_model.pkl saved! Copy to backend/ folder.")

DROUGHT MODEL RESULTS
F1-Score : 0.7925
ROC-AUC  : 0.9707

              precision    recall  f1-score   support

  No Drought       0.94      0.97      0.95       800
     Drought       0.86      0.73      0.79       200

    accuracy                           0.92      1000
   macro avg       0.90      0.85      0.87      1000
weighted avg       0.92      0.92      0.92      1000

drought_model.pkl saved! Copy to backend/ folder.
